Fidel CRNN checkpoint evaluated on AHL-D, no retraining.

In [ ]:
#@title Test Fidel CRNN on AHL-D 29K

!pip install -q datasets jiwer

import os
import json
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from jiwer import cer, wer

from google.colab import drive

drive.mount("/content/drive")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/"
    "amharic_htr_research/ocr/research/crnn/"
    "extended/best_checkpoint.pt"
)

OUTPUT_PATH = (
    "/content/drive/MyDrive/"
    "amharic_htr_research/ocr/research/crnn/"
    "ahld29k_predictions.csv"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

state_dict = checkpoint["model_state_dict"]

char_to_idx = checkpoint["char_to_idx"]

idx_to_char = {
    int(k): v
    for k, v in checkpoint["idx_to_char"].items()
}

blank_idx = checkpoint["blank_idx"]


class CRNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1))
        )

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.2
        )

        self.classifier = nn.Linear(
            512,
            num_classes
        )

    def forward(self, x):

        x = self.cnn(x)

        x = F.adaptive_avg_pool2d(
            x,
            (1, x.size(3))
        )

        x = x.squeeze(2)

        x = x.permute(2, 0, 1)

        x, _ = self.rnn(x)

        return self.classifier(x)


model = CRNN(
    num_classes=len(char_to_idx) + 1
).to(DEVICE)

model.load_state_dict(
    state_dict
)

model.eval()

print(
    f"Model loaded | best epoch {checkpoint.get('epoch')} | "
    f"Fidel CER {checkpoint.get('best_cer'):.4f}"
)

ahld = load_dataset(
    "misiker/AHLD-29k",
    split="train"
)

print(
    f"AHL-D loaded | {len(ahld)} samples"
)

image_column = None
text_column = None

for column in ahld.column_names:

    if column.lower() in [
        "image",
        "img",
        "picture"
    ]:
        image_column = column
        break

for column in ahld.column_names:

    if column.lower() in [
        "text",
        "label",
        "transcription",
        "transcript",
        "sentence",
        "gt"
    ]:
        text_column = column
        break

if image_column is None:
    raise RuntimeError(
        f"Image column not found: {ahld.column_names}"
    )

if text_column is None:
    raise RuntimeError(
        f"Text column not found: {ahld.column_names}"
    )

print(
    f"Columns | image={image_column} | text={text_column}"
)


class AHLDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data[idx]

        image = row[image_column]
        text = str(row[text_column])

        if not isinstance(image, Image.Image):
            image = Image.fromarray(
                np.asarray(image)
            )

        image = image.convert("L")

        image = image.resize(
            (512, 64),
            Image.Resampling.LANCZOS
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        return image, text


def collate_fn(batch):

    images = torch.stack(
        [x[0] for x in batch]
    )

    texts = [
        x[1]
        for x in batch
    ]

    return images, texts


loader = DataLoader(
    AHLDataset(ahld),
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)


def greedy_decode(outputs):

    predictions = outputs.argmax(dim=2)

    results = []

    for sequence in predictions.permute(1, 0):

        text = []
        previous = blank_idx

        for index in sequence.tolist():

            if (
                index != blank_idx
                and index != previous
            ):
                text.append(
                    idx_to_char.get(
                        index,
                        ""
                    )
                )

            previous = index

        results.append(
            "".join(text)
        )

    return results


references = []
predictions = []

print("Running inference...")

with torch.no_grad():

    for images, texts in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(images)

        batch_predictions = greedy_decode(
            outputs
        )

        references.extend(texts)
        predictions.extend(batch_predictions)

ahld_cer = cer(
    references,
    predictions
)

ahld_wer = wer(
    references,
    predictions
)

results = pd.DataFrame({
    "ground_truth": references,
    "prediction": predictions
})

results.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print(
    f"AHL-D 29K | CER {ahld_cer:.4f} | WER {ahld_wer:.4f}"
)

for i in range(
    min(10, len(results))
):

    print(
        f"GT: {references[i]}"
    )

    print(
        f"PR: {predictions[i]}"
    )

print(
    f"Saved: {OUTPUT_PATH}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded | best epoch 13 | Fidel CER 0.1647


data/train-00004-of-00007.parquet: reconstructing file:   0%|          |  0.00B /  549MB            

data/train-00004-of-00007.parquet: downloading bytes:           |  0.00B            

data/train-00005-of-00007.parquet: reconstructing file:   0%|          |  0.00B /  533MB            

data/train-00005-of-00007.parquet: downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/3046 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2990 [00:00<?, ? examples/s]

AHL-D loaded | 23911 samples
Columns | image=image | text=text
Running inference...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


AHL-D 29K | CER 0.4118 | WER 0.7546
GT: አንቀጽ 1፡ ፍጥረት፣ ጥንት፣ ሕግ፣ ነቢያት፣ ነገሥታት፣ ስደት
PR: ንቀድ  ፍጥረት  ፣ ጥንት  ፣ ሕግ ፣ ኃቢያት  ፣ ነሥታት  ፣ ስዶት  

GT: በመጀመሪያ ፈጣሪ ሰማይንና ምድርን ፈጠረ፤ እንዲሁም አዳምንና ሔዋንን በራሱ
PR: መጀመሪያ  ፈጣሪ  ሰማይንና  ምድርጎ  ፈጠሪ  ፣ እንዲሁም  አዳምንና  ሔዋንን  በረርሰ  

GT: አምሳል ፈጥሮ በዔድን ገነት አኖራቸው። ነገር ግን በእባብ ተታለው ትእዛዙን
PR: አምሳል  ፈጥሮ  በጼድን  ገነት  አናደባቸው።  ነገር ግን በአገብ  ቲታለው  ትእዛዙ  ፣ 

GT: ሲጥሱ ከገነት ተባረሩ፤ ኀዘንና መከራ የምድር ዕጣ ፈንታቸው ሆነ። ዘመናት ካለፉ
PR: ብሱ  ከገነት  ተባረሩ  ፥ ካዘንና  መከራ  የምድር  ዕጣ ፈንታቸው  ሆነ።  ዘመናት  ካለት

GT: በኋላ ፣ እግዚአብሔር ሙሴና በሲና ተራራ ላይ ጠርቶ ሕግጋትን ስጠው፤ ዐሥርቱ
PR: አላ  ፣ እግዚአብሔር  ሙሴና  በስና  ተራራ  ላይ ጠርቶ  ሕግጋትን  ሰጠው  ፤ መርቱ

GT: ትእዛዛት በጽላት ተጻፉ። ዳሩ ግን ሕዝቡ በተደጋጋሚ ልባቸውን እልከኛ በማድረግ
PR: ከላዛት  በድላት  ተጻፋ።  ዳሩ ግን ሕዝቡ  በተደጋጋሚ  ልባቸውት  እልከኛ  በማድረግ  

GT: የሚናገረውን ለመስማት ቸል አሉ። እግዚአብሔር በበረሀ ጉዞአቸው በተአምራት
PR: ሚናገረውን  ለመስማት  ችል አሉ።  እግዚአብሔበር  በበረህ  ገዞአቸው  በተአምራት 

GT: ቢረዳቸውም እነርሱ ከአምልኮቱ ራቁ። ነቢያት ተነስተው የእግዚአብሔርን ቃል
PR: ረዳቸውም  እነርሱ  ከአምልስቱ  ረራቁ።  ነቢያት  ተነስተው  የአግዚአብሔርን  ቃር
GT: ቢያደርሱም እንደ መዝሙረኛው ዳዊትና ጥበበኛው ሰሎሞን ያሉ መሪዎች ቢነሱም
PR: ያደርሱም  እንደ 